# SpikeFormer Training - Kaggle GPU

Trains both **ANN Transformer** (baseline) and **SNN SpikeFormer** models
on CIFAR-10 using Kaggle's 30h/week GPU quota.

Reference: [Xpikeformer paper (arXiv:2408.08794v2)](https://arxiv.org/abs/2408.08794v2)

## Setup

In [ ]:
# Clone SpikeFormer repo (if starting fresh on Kaggle)
!rm -rf SpikeFormer 2>/dev/null || true
!git clone https://github.com/yourusername/SpikeFormer.git
!cd SpikeFormer && pip install -e . -q

In [ ]:
import os
import sys
sys.path.insert(0, '/root/SpikeFormer' if os.path.exists('/root/SpikeFormer') else '.')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from src.ann.transformer import CIFAR10ANNTransformer, create_ann_transformer
from src.snn.spikeformer import CIFAR10Spikeformer, create_spikeformer

# Verify GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Configuration

In [ ]:
# Training hyperparameters
class Config:
    # Model selection
    model_type = 'both'  # 'ann', 'snn', or 'both'
    
    # Data
    batch_size = 128
    num_workers = 2
    
    # Training
    num_epochs = 100
    learning_rate = 1e-3
    weight_decay = 1e-4
    
    # LR schedule
    lr_warmup_epochs = 5
    lr_max = 1e-3
    lr_min = 1e-5
    
    # SNN-specific
    timesteps = 4
    
    # Checkpointing
    save_every = 10  # epochs
    
    # Paths - separate directories for ANN vs SNN
    ann_checkpoint_dir = '/root/SpikeFormer/checkpoints_ann'
    snn_checkpoint_dir = '/root/SpikeFormer/checkpoints'

config = Config()
os.makedirs(config.ann_checkpoint_dir, exist_ok=True)
os.makedirs(config.snn_checkpoint_dir, exist_ok=True)

print(f"ANN checkpoints: {config.ann_checkpoint_dir}")
print(f"SNN checkpoints: {config.snn_checkpoint_dir}")

## Data Loading

In [ ]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, num_workers=config.num_workers)
test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False, num_workers=config.num_workers)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

## Model Creation

In [ ]:
def create_models():
    """Create ANN and/or SNN models based on config."""
    models = {}
    
    if config.model_type in ['ann', 'both']:
        ann_model = create_ann_transformer().to(device)
        models['ann'] = ann_model
        print(f"ANN Transformer: {sum(p.numel() for p in ann_model.parameters()):,} params")
    
    if config.model_type in ['snn', 'both']:
        snn_model = create_spikeformer(timesteps=config.timesteps).to(device)
        models['snn'] = snn_model
        print(f"SNN SpikeFormer: {sum(p.numel() for p in snn_model.parameters()):,} params")
    
    return models

models = create_models()

## Training Functions

In [ ]:
def get_lr(epoch):
    """Cosine annealing LR schedule with warmup."""
    if epoch < config.lr_warmup_epochs:
        return config.lr_max * (epoch + 1) / config.lr_warmup_epochs
    else:
        progress = (epoch - config.lr_warmup_epochs) / (config.num_epochs - config.lr_warmup_epochs)
        return config.lr_min + (config.lr_max - config.lr_min) * 0.5 * (1 + np.cos(np.pi * progress))

import numpy as np

def train_epoch(model, loader, optimizer, criterion, device, is_snn=False):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_idx, (data, target) in enumerate(loader):
        data, target = data.to(device), target.to(device)
        
        optimizer.zero_grad()
        
        if is_snn:
            # SNN: encode input over T timesteps
            output = model(data, timesteps=config.timesteps)
        else:
            output = model(data)
        
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = output.max(1)
        total += target.size(0)
        correct += predicted.eq(target).sum().item()
        
        if batch_idx % 100 == 0:
            print(f'  Batch {batch_idx}/{len(loader)}: Loss={loss.item():.4f}')
    
    return total_loss / len(loader), 100. * correct / total

def evaluate(model, loader, device, is_snn=False):
    """Evaluate model on test set."""
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            
            if is_snn:
                output = model(data, timesteps=config.timesteps)
            else:
                output = model(data)
            
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()
    
    return 100. * correct / total

## Training Loop

In [ ]:
def train_model(model_name, model, checkpoint_dir):
    """Train a single model."""
    print(f"\n{'='*50}")
    print(f"Training {model_name}")
    print(f"{'='*50}")
    
    is_snn = model_name == 'snn'
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    
    best_acc = 0
    
    for epoch in range(config.num_epochs):
        # Update LR
        lr = get_lr(epoch)
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr
        
        print(f"\nEpoch {epoch+1}/{config.num_epochs} (LR={lr:.6f})")
        
        # Train
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device, is_snn)
        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
        
        # Evaluate
        test_acc = evaluate(model, test_loader, device, is_snn)
        print(f"Test Acc: {test_acc:.2f}%")
        
        # Save checkpoint
        if (epoch + 1) % config.save_every == 0 or test_acc > best_acc:
            ckpt_path = os.path.join(checkpoint_dir, f'{model_name}_epoch_{epoch+1}.pth')
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'test_acc': test_acc,
            }, ckpt_path)
            print(f"Saved checkpoint: {ckpt_path}")
        
        if test_acc > best_acc:
            best_acc = test_acc
            best_path = os.path.join(checkpoint_dir, f'{model_name}_best.pth')
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'test_acc': test_acc,
            }, best_path)
            print(f"New best! Saved: {best_path}")
    
    return best_acc

# Train selected models
results = {}
for model_name, model in models.items():
    checkpoint_dir = config.ann_checkpoint_dir if model_name == 'ann' else config.snn_checkpoint_dir
    results[model_name] = train_model(model_name, model, checkpoint_dir)

print(f"\n{'='*50}")
print("Training Complete!")
print(f"{'='*50}")
for model_name, acc in results.items():
    print(f"{model_name.upper()} Best Test Accuracy: {acc:.2f}%")

## Save Final Models

In [ ]:
# Save final models for benchmarking
for model_name, model in models.items():
    checkpoint_dir = config.ann_checkpoint_dir if model_name == 'ann' else config.snn_checkpoint_dir
    final_path = os.path.join(checkpoint_dir, f'{model_name}_final.pth')
    torch.save({
        'model_state_dict': model.state_dict(),
        'test_acc': results.get(model_name, 0),
        'config': {
            'timesteps': config.timesteps if model_name == 'snn' else None,
        }
    }, final_path)
    print(f"Final model saved: {final_path}")

# List checkpoints
print("\nANN Checkpoints:")
for f in sorted(os.listdir(config.ann_checkpoint_dir)):
    print(f"  {f}")
print("\nSNN Checkpoints:")
for f in sorted(os.listdir(config.snn_checkpoint_dir)):
    print(f"  {f}")

## Download Checkpoints (for local benchmarking)

In [ ]:
# Zip checkpoints for download
!cd /root/SpikeFormer && zip -r ann_checkpoints.zip checkpoints_ann/
!cd /root/SpikeFormer && zip -r snn_checkpoints.zip checkpoints/
print("Checkpoints zipped for download")